In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp, col, count, sum as _sum, 
    avg, round as _round, min as _min, 
    max as _max, window, desc, asc
)

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")


Spark 4.0.0-preview2 — gotowy


In [12]:
df = spark.read.json("L2/transactions_10k.jsonl")

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"Liczba rekordów: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |ży

In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'


root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [13]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [7]:
store_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("category")
)
store_summary.show()

+-----------+---------+----------+-----------+
|   category|liczba_tx|  suma_PLN|srednia_PLN|
+-----------+---------+----------+-----------+
|elektronika|     2542|1520770.69|     598.26|
|    książki|     2574| 851382.08|     330.76|
|     odzież|     2453| 849877.55|     346.46|
|    żywność|     2431| 789514.43|     324.77|
+-----------+---------+----------+-----------+



In [8]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [9]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))    # okno 1-godzinne + krok 30 min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [10]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [21]:
### ZADANIE 1 - Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

(
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("avg_amt"))
    .select(col("window.start").alias("od"),
            col("window.end").alias("do"),
            "avg_amt")
    .orderBy(asc("avg_amt"))
    .show()
)

+-------------------+-------------------+-------+
|                 od|                 do|avg_amt|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00| 395.01|
|2026-04-12 10:00:00|2026-04-12 11:00:00| 412.92|
|2026-04-12 09:00:00|2026-04-12 10:00:00| 415.91|
+-------------------+-------------------+-------+



In [19]:
### ZADANIE 2 - Policz ile transakcji per kategoria było w oknie 09:00–09:30.

(
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("tx_count"))
    .filter(col("window.start").cast("string").contains("09:00:00"))
    .select(col("window.start").alias("od"), 
            col("window.end").alias("do"),
            "category", "tx_count")
    .orderBy(desc("tx_count"))
    .show()
)




+-------------------+-------------------+-----------+--------+
|                 od|                 do|   category|tx_count|
+-------------------+-------------------+-----------+--------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|    książki|     622|
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|     611|
|2026-04-12 09:00:00|2026-04-12 09:30:00|     odzież|     605|
|2026-04-12 09:00:00|2026-04-12 09:30:00|    żywność|     567|
+-------------------+-------------------+-----------+--------+



In [16]:
### ZADANIE 2 - Policz ile transakcji per kategoria było w oknie 09:00–09:30.

(
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("tx_count"))
    .filter(col("window_start").cast("string").contains("09:00:00"))
    .orderBy(desc("tx_count"))
    .show()
)




{"ts": "2026-05-10 10:35:06,833", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `window_start` cannot be resolved. Did you mean one of the following? [`window`, `tx_count`, `category`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor47.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o201.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `window_start` cannot be resolved. Did you mean one of the following? [`window`, `tx_count`, `category`]. SQLSTATE: 42703;\n'Filter 'contains(cast('window_start as string), 09:00:00)\n+- Aggregate [window#173, category#9], [window#173, category#9, count(tx_id#12) AS tx_count#171L]\n   +

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `window_start` cannot be resolved. Did you mean one of the following? [`window`, `tx_count`, `category`]. SQLSTATE: 42703;
'Filter 'contains(cast('window_start as string), 09:00:00)
+- Aggregate [window#173, category#9], [window#173, category#9, count(tx_id#12) AS tx_count#171L]
   +- Project [named_struct(start, knownnullable(precisetimestampconversion(((precisetimestampconversion(timestamp#20, TimestampType, LongType) - CASE WHEN (((precisetimestampconversion(timestamp#20, TimestampType, LongType) - 0) % 1800000000) < cast(0 as bigint)) THEN (((precisetimestampconversion(timestamp#20, TimestampType, LongType) - 0) % 1800000000) + 1800000000) ELSE ((precisetimestampconversion(timestamp#20, TimestampType, LongType) - 0) % 1800000000) END) - 0), LongType, TimestampType)), end, knownnullable(precisetimestampconversion((((precisetimestampconversion(timestamp#20, TimestampType, LongTy

In [25]:
### ZADANIE 3 - Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

(
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .select(col("window.start").alias("od"),
            col("window.end").alias("do"),
            "tx_count")
    .orderBy(desc("tx_count"))
    .show()
)

    

+-------------------+-------------------+--------+
|                 od|                 do|tx_count|
+-------------------+-------------------+--------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|    1234|
|2026-04-12 09:00:00|2026-04-12 09:15:00|    1171|
|2026-04-12 09:30:00|2026-04-12 09:45:00|    1156|
|2026-04-12 08:45:00|2026-04-12 09:00:00|    1139|
|2026-04-12 09:45:00|2026-04-12 10:00:00|    1100|
|2026-04-12 08:30:00|2026-04-12 08:45:00|     899|
|2026-04-12 10:00:00|2026-04-12 10:15:00|     858|
|2026-04-12 08:15:00|2026-04-12 08:30:00|     644|
|2026-04-12 10:15:00|2026-04-12 10:30:00|     582|
|2026-04-12 08:00:00|2026-04-12 08:15:00|     468|
|2026-04-12 10:30:00|2026-04-12 10:45:00|     443|
|2026-04-12 10:45:00|2026-04-12 11:00:00|     306|
+-------------------+-------------------+--------+

